# Training Dynamics: Dominance and Agreement Subset

Dominance and representation-decoding agreement over normalized training progress for the model families shown in the main-paper training-dynamics figure.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "vis.py").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from vis import save_matplotlib_figure_bundle, set_matplotlib_paper_font

CACHE_DIR = REPO_ROOT / ".analysis_cache" / "training_dynamics"
FIG_PATH = REPO_ROOT / "figs" / "subsets" / "training_dynamics_dominance_agreement_subset"
MODEL_FAMILIES = ("OLMo-2-7B", "Apertus-8B")
LAYER_BINS = ("early", "middle", "late")
LAYER_COLORS = {
    "early": "#0072B2",
    "middle": "#E69F00",
    "late": "#CC79A7",
}
PANEL_SPECS = (
    ("Dominance", (("repr_dominance", "repr"), ("dec_dominance", "dec"))),
    ("Representation-decoding agreement", (("agreement_repr_vs_dec", "agreement"),)),
)
ESTIMATOR_STYLES = {
    "repr": {"linestyle": "-", "linewidth": 2.2},
    "dec": {"linestyle": "--", "linewidth": 2.2},
    "agreement": {"linestyle": "-", "linewidth": 2.2},
}
TICK_FONT_SIZE = 14
PANEL_TITLE_FONT_SIZE = 16
ROW_LABEL_FONT_SIZE = 16
LEGEND_FONT_SIZE = 16
FIGURE_TITLE_FONT_SIZE = 19
AXIS_LABEL_FONT_SIZE = 16

set_matplotlib_paper_font()


In [ ]:
cache_paths = sorted(
    CACHE_DIR.glob("training_progress_plot_summary_*.parquet"),
    key=lambda path: path.stat().st_mtime_ns,
)
if not cache_paths:
    raise FileNotFoundError(
        "Missing the training-progress plot cache. Execute 05_training_dynamics.ipynb first."
    )

training_summary = pd.read_parquet(cache_paths[-1])
panel_metrics = [
    metric_name
    for _, metric_specs in PANEL_SPECS
    for metric_name, _ in metric_specs
]
subset_summary = training_summary[
    training_summary["bin_scheme"].eq("early_middle_late")
    & training_summary["training_model_family"].isin(MODEL_FAMILIES)
    & training_summary["metric"].isin(panel_metrics)
    & training_summary["training_layer_bin"].isin(LAYER_BINS)
].copy()

expected_pairs = {
    (family, metric_name, layer_bin)
    for family in MODEL_FAMILIES
    for metric_name in panel_metrics
    for layer_bin in LAYER_BINS
}
available_pairs = set(
    subset_summary[["training_model_family", "metric", "training_layer_bin"]]
    .drop_duplicates()
    .itertuples(index=False, name=None)
)
missing_pairs = sorted(expected_pairs - available_pairs)
if missing_pairs:
    raise ValueError(f"Missing requested training-dynamics series: {missing_pairs}")

subset_summary.head()


In [ ]:
fig, axes = plt.subplots(
    len(MODEL_FAMILIES),
    len(PANEL_SPECS),
    figsize=(10.0, 5.9),
    squeeze=False,
    sharex=True,
    sharey=True,
    gridspec_kw={"hspace": 0.22, "wspace": 0.12},
)
for row_idx, family in enumerate(MODEL_FAMILIES):
    family_df = subset_summary[
        subset_summary["training_model_family"].eq(family)
    ]
    for col_idx, (panel_label, metric_specs) in enumerate(PANEL_SPECS):
        ax = axes[row_idx, col_idx]
        for metric_name, estimator_key in metric_specs:
            metric_df = family_df[family_df["metric"].eq(metric_name)]
            style = ESTIMATOR_STYLES[estimator_key]
            for layer_bin in LAYER_BINS:
                series = metric_df[
                    metric_df["training_layer_bin"].eq(layer_bin)
                ].sort_values("training_progress")
                ax.plot(
                    series["training_progress"],
                    series["value"],
                    color=LAYER_COLORS[layer_bin],
                    marker="o",
                    markersize=3.8,
                    **style,
                )
        ax.set_xlim(-0.02, 1.02)
        ax.set_ylim(0.0, 1.0)
        ax.set_yticks([0.0, 0.25, 0.5, 0.75, 1.0])
        ax.grid(True, linewidth=0.45, alpha=0.25)
        ax.tick_params(labelsize=TICK_FONT_SIZE)
        if row_idx == 0:
            ax.set_title(
                panel_label, fontsize=PANEL_TITLE_FONT_SIZE, fontweight=600, pad=9
            )
        if col_idx == 0:
            ax.set_ylabel(
                family, fontsize=ROW_LABEL_FONT_SIZE, fontweight=600, labelpad=11
            )
        else:
            ax.tick_params(axis="y", left=False, labelleft=False)

layer_handles = [
    plt.Line2D([], [], color="none", linestyle="none", label="Layer bin:"),
    *[
        plt.Line2D(
            [0], [0], color=LAYER_COLORS[layer_bin], linewidth=3.0, label=layer_bin.title()
        )
        for layer_bin in LAYER_BINS
    ],
]
estimator_handles = [
    plt.Line2D([], [], color="none", linestyle="none", label="Estimator:"),
    plt.Line2D([0], [0], color="#333333", linestyle="-", linewidth=2.5, label="Representation probe"),
    plt.Line2D([0], [0], color="#333333", linestyle="--", linewidth=2.5, label="Raw LogitLens Top-p"),
]
layer_legend = fig.legend(
    handles=layer_handles,
    loc="lower center",
    bbox_to_anchor=(0.55, 0.070),
    ncol=len(layer_handles),
    frameon=False,
    fontsize=LEGEND_FONT_SIZE,
    handlelength=1.8,
    columnspacing=0.9,
)
estimator_legend = fig.legend(
    handles=estimator_handles,
    loc="lower center",
    bbox_to_anchor=(0.55, 0.010),
    ncol=len(estimator_handles),
    frameon=False,
    fontsize=LEGEND_FONT_SIZE,
    handlelength=1.8,
    columnspacing=0.9,
)
layer_legend.get_texts()[0].set_fontweight(600)
estimator_legend.get_texts()[0].set_fontweight(600)

fig.suptitle(
    "Dominance and Agreement Across Pretraining on PUD21",
    fontsize=FIGURE_TITLE_FONT_SIZE,
    fontweight=600,
    x=0.55,
    y=0.975,
)
fig.supxlabel(
    "Normalized training progress",
    fontsize=AXIS_LABEL_FONT_SIZE,
    fontweight=600,
    x=0.55,
    y=0.160,
)
fig.subplots_adjust(left=0.11, right=0.99, bottom=0.26, top=0.85)
save_matplotlib_figure_bundle(fig, FIG_PATH)
plt.show()
